<a href="https://colab.research.google.com/github/Andreea1605/VAT-Identifier-Discovery/blob/main/VAT_Identifier_Discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/BasicCompanyDataAsOneFile-2026-08-01.csv"
chunk_size = 250000

print("STEP 1: Extracting the first 10 records for visual inspection")
# Read only 10 rows but keep all columns to get an overview of the dataset structure
preview_df = pd.read_csv(file_path, nrows=10, low_memory=False)
preview_df.to_csv("preview_10_records.csv", index=False)
print("Saved 'preview_10_records.csv'\n")

STEP 1: Extracting the first 10 records for visual inspection
Saved 'preview_10_records.csv'



In [3]:
print("STEP 2: Data Profiling & Domain Discovery")
print("Reading the massive file in chunks to prevent memory overload")

# Define strictly the required columns to free up RAM
required_columns = ['CompanyName', ' CompanyNumber', 'CompanyStatus', 'SICCode.SicText_1']
active_companies = []

total_active = 0
total_empty = 0
total_none_supplied = 0
unique_industries = set()  # Set for automatically storing unique domains

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False, usecols=required_columns):
    # Filter only active companies
    active_chunk = chunk[chunk['CompanyStatus'] == 'Active']

    # Aggregate data for profiling: update counters only for active companies
    total_active += len(active_chunk)
    total_empty += active_chunk['SICCode.SicText_1'].isna().sum()
    total_none_supplied += active_chunk['SICCode.SicText_1'].str.contains("None Supplied", case=False, na=False).sum()

    # Extract unique industries for this chunk (ignoring NaN and 'None Supplied')
    valid_sic = active_chunk['SICCode.SicText_1'].dropna()
    valid_sic = valid_sic[~valid_sic.str.contains("None Supplied", case=False)]
    unique_industries.update(valid_sic.unique())

    # Keep the filtered chunk for sampling
    active_companies.append(active_chunk)

# Convert the unique set to a sorted list and get the total count
industries_list = sorted(list(unique_industries))
total_unique_industries = len(industries_list)

print("\n--- DATA PROFILING RESULTS ---")
print(f"Total active companies found: {total_active:,}")
print(f"Companies with missing (empty) domain: {total_empty:,}")
print(f"Companies with domain declared as 'None Supplied': {total_none_supplied:,}")

total_dirty = total_empty + total_none_supplied
dirty_percentage = (total_dirty / total_active) * 100 if total_active > 0 else 0
print(f"In total, {dirty_percentage:.2f}% of the active companies have invalid data in the activity category column.")
print(f"Total unique valid industries discovered: {total_unique_industries:,}\n")

# Save the full list of unique industries for documentation
pd.DataFrame(industries_list, columns=['Industry (SIC Code)']).to_csv("all_unique_industries.csv", index=False)
print("Saved 'all_unique_industries.csv'\n")

STEP 2: Data Profiling & Domain Discovery
Reading the massive file in chunks to prevent memory overload

--- DATA PROFILING RESULTS ---
Total active companies found: 5,190,464
Companies with missing (empty) domain: 0
Companies with domain declared as 'None Supplied': 216,285
In total, 4.17% of the active companies have invalid data in the activity category column.
Total unique valid industries discovered: 991

Saved 'all_unique_industries.csv'



In [4]:
print("STEP 3: Data Cleaning & Stratified Sampling")
print("Concatenating processed chunks")
df_active = pd.concat(active_companies)

print("Cleaning data")
# Remove empty values (NaN)
df_active = df_active.dropna(subset=['SICCode.SicText_1'])
# Remove 'None Supplied' anomalies
df_active = df_active[~df_active['SICCode.SicText_1'].str.contains("None Supplied", case=False, na=False)]

print("Identifying the top 10 largest industries in the UK")
top_10_industries = df_active['SICCode.SicText_1'].value_counts().head(10).index

# Keep only the companies operating in these top 10 industries
df_top_10 = df_active[df_active['SICCode.SicText_1'].isin(top_10_industries)]

print("Generating the stratified sample (10 companies x 10 industries)")
# Group by industry and randomly sample 10 from each group
final_sample = df_top_10.groupby('SICCode.SicText_1').sample(n=10, random_state=36)

final_sample.to_csv("sample_100_companies_final.csv", index=False)

print("\nSUCCESS! The 'sample_100_companies_final.csv' file has been created.")
print("The final distribution of sample by industry is:")
print(final_sample['SICCode.SicText_1'].value_counts())

STEP 3: Data Cleaning & Stratified Sampling
Concatenating processed chunks
Cleaning data
Identifying the top 10 largest industries in the UK
Generating the stratified sample (10 companies x 10 industries)

SUCCESS! The 'sample_100_companies_final.csv' file has been created.
The final distribution of sample by industry is:
SICCode.SicText_1
41100 - Development of building projects                                     10
47910 - Retail sale via mail order houses or via Internet                    10
62020 - Information technology consultancy activities                        10
64209 - Activities of other holding companies n.e.c.                         10
68100 - Buying and selling of own real estate                                10
68209 - Other letting and operating of own or leased real estate             10
70229 - Management consultancy activities other than financial management    10
82990 - Other business support service activities n.e.c.                     10
96090 - Other serv

In [5]:
df_active = pd.concat(active_companies)
df_active = df_active.dropna(subset=['SICCode.SicText_1'])
df_active = df_active[~df_active['SICCode.SicText_1'].str.contains("None Supplied", case=False, na=False)]

account_col = pd.read_csv(file_path, usecols=[' CompanyNumber', 'Accounts.AccountCategory'], low_memory=False)
df_active = df_active.merge(account_col, on=' CompanyNumber', how='left')

print(df_active['Accounts.AccountCategory'].value_counts())


Accounts.AccountCategory
MICRO ENTITY                   1661012
TOTAL EXEMPTION FULL           1206177
NO ACCOUNTS FILED              1189517
DORMANT                         569666
UNAUDITED ABRIDGED              147656
FULL                             70342
SMALL                            61706
AUDIT EXEMPTION SUBSIDIARY       32576
GROUP                            25842
MEDIUM                            6102
TOTAL EXEMPTION SMALL             1431
AUDITED ABRIDGED                  1095
ACCOUNTS TYPE NOT AVAILABLE        651
FILING EXEMPTION SUBSIDIARY        396
PARTIAL EXEMPTION                   10
Name: count, dtype: int64


In [6]:
LARGE_ACCOUNT_CATEGORIES = ['FULL', 'GROUP', 'MEDIUM']

df_large = df_active[df_active['Accounts.AccountCategory'].isin(LARGE_ACCOUNT_CATEGORIES)]
print(f"Remaining large companies: {len(df_large):,}")
print(df_large['Accounts.AccountCategory'].value_counts())

top_10_industries_large = df_large['SICCode.SicText_1'].value_counts().head(10).index
print("\nTop 10 industries (large companies only):")
print(top_10_industries_large.tolist())

df_top_10_large = df_large[df_large['SICCode.SicText_1'].isin(top_10_industries_large)]

final_sample_large = df_top_10_large.groupby('SICCode.SicText_1').sample(n=10, random_state=36)
final_sample_large.to_csv("sample_100_large_companies.csv", index=False)

print("\nSaved 'sample_100_large_companies.csv'")
print(final_sample_large['SICCode.SicText_1'].value_counts())


Remaining large companies: 102,286
Accounts.AccountCategory
FULL      70342
GROUP     25842
MEDIUM     6102
Name: count, dtype: int64

Top 10 industries (large companies only):
['64209 - Activities of other holding companies n.e.c.', '70100 - Activities of head offices', '82990 - Other business support service activities n.e.c.', '64999 - Financial intermediation not elsewhere classified', '68209 - Other letting and operating of own or leased real estate', '41100 - Development of building projects', '96090 - Other service activities n.e.c.', '74990 - Non-trading company', '64205 - Activities of financial services holding companies', '35110 - Production of electricity']

Saved 'sample_100_large_companies.csv'
SICCode.SicText_1
35110 - Production of electricity                                   10
41100 - Development of building projects                            10
64205 - Activities of financial services holding companies          10
64209 - Activities of other holding companies n.e.c

In [7]:
NON_OPERATING_KEYWORDS = [
    'holding compan', 'head office', 'non-trading', 'financial intermediation',
    'letting and operating', 'financial services holding',
]

def is_operating_business(sic_text):
    if pd.isna(sic_text):
        return False
    return not any(keyword in sic_text.lower() for keyword in NON_OPERATING_KEYWORDS)

df_operating = df_large[df_large['SICCode.SicText_1'].apply(is_operating_business)]
print(f"Large + operating businesses: {len(df_operating):,}")

top_10_operating = df_operating['SICCode.SicText_1'].value_counts().head(10).index
print("\nTop 10 industries (large + operating only):")
print(top_10_operating.tolist())

df_top_10_operating = df_operating[df_operating['SICCode.SicText_1'].isin(top_10_operating)]

final_sample_operating = df_top_10_operating.groupby('SICCode.SicText_1').sample(n=10, random_state=36)

SAVE_PATH = "/content/drive/MyDrive/sample_100_operating_large_companies.csv"
final_sample_operating.to_csv(SAVE_PATH, index=False)

print(f"\nSaved to {SAVE_PATH}")
print(final_sample_operating['SICCode.SicText_1'].value_counts())

Large + operating businesses: 72,976

Top 10 industries (large + operating only):
['82990 - Other business support service activities n.e.c.', '41100 - Development of building projects', '96090 - Other service activities n.e.c.', '35110 - Production of electricity', '85200 - Primary education', '62090 - Other information technology service activities', '55100 - Hotels and similar accommodation', '65120 - Non-life insurance', '68100 - Buying and selling of own real estate', '62012 - Business and domestic software development']

Saved to /content/drive/MyDrive/sample_100_operating_large_companies.csv
SICCode.SicText_1
35110 - Production of electricity                           10
41100 - Development of building projects                    10
55100 - Hotels and similar accommodation                    10
62012 - Business and domestic software development          10
62090 - Other information technology service activities     10
65120 - Non-life insurance                                  1

In [8]:
print(final_sample_operating.columns.tolist())

['CompanyName', ' CompanyNumber', 'CompanyStatus', 'SICCode.SicText_1', 'Accounts.AccountCategory']


In [9]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, unquote

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

IGNORE_DOMAINS = [
    "find-and-update.company-information.service.gov.uk",
    "endole.co.uk", "companieshouse.gov.uk", "vat-search.co.uk",
    "vatverifier.com", "linkedin.com", "facebook.com", "wikipedia.org",
    "bloomberg.com", "opencorporates.com", "creditsafe.com",
]

VAT_PATTERN = re.compile(r"\bGB\s?(\d{3})\s?(\d{4})\s?(\d{2})(\d{3})?\b", re.IGNORECASE)
CANDIDATE_PAGES = ["", "/terms", "/terms-and-conditions", "/legal", "/legal-info",
                    "/about", "/contact", "/privacy-policy"]


def find_company_website(company_name: str):
    query = f"{company_name} official website"
    try:
        resp = requests.get(
            "https://html.duckduckgo.com/html/",
            params={"q": query}, headers=HEADERS, timeout=10
        )
        soup = BeautifulSoup(resp.text, "html.parser")
        for link in soup.select("a.result__a"):
            href = link.get("href", "")

            if "uddg=" in href:
                parsed = urlparse("https:" + href if href.startswith("//") else href)
                qs = parse_qs(parsed.query)
                if "uddg" in qs:
                    real_url = unquote(qs["uddg"][0])
                else:
                    continue
            elif href.startswith("http"):
                real_url = href
            else:
                continue

            if not any(d in real_url for d in IGNORE_DOMAINS):
                domain = re.match(r"https?://(?:www\.)?([^/]+)", real_url)
                if domain:
                    return f"https://{domain.group(1)}"
    except requests.RequestException:
        pass
    return None


def find_vat_candidates(base_url: str):
    candidates = set()
    for path in CANDIDATE_PAGES:
        url = base_url.rstrip("/") + path
        try:
            resp = requests.get(url, headers=HEADERS, timeout=8)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, "html.parser")
            text = soup.get_text(" ")
            for match in VAT_PATTERN.finditer(text):
                raw = "".join(g for g in match.groups() if g)
                candidates.add("GB" + raw)
        except requests.RequestException:
            continue
        time.sleep(1)
    return list(candidates)


def run_discovery(sample_csv: str, output_csv: str):
    df = pd.read_csv(sample_csv)
    df.columns = df.columns.str.strip()

    results = []
    for _, row in df.iterrows():
        name = row["CompanyName"]
        print(f"Processing: {name}")

        website = find_company_website(name)
        if not website:
            results.append({"CompanyName": name, "CompanyNumber": row.get("CompanyNumber"),
                             "website": None, "vat_candidates": None,
                             "status": "no_website_found"})
            time.sleep(2)
            continue

        candidates = find_vat_candidates(website)
        results.append({
            "CompanyName": name,
            "CompanyNumber": row.get("CompanyNumber"),
            "website": website,
            "vat_candidates": "; ".join(candidates) if candidates else None,
            "status": "candidate_found_NEEDS_HMRC_VERIFICATION" if candidates else "not_found",
        })
        time.sleep(2)

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"\nSaved {output_csv}")
    print(out_df["status"].value_counts())
    return out_df

# Manual isolated test before running on full sample
print(find_company_website("THE DONKEY SANCTUARY"))

results_df = run_discovery(
    "/content/drive/MyDrive/sample_100_operating_large_companies.csv",
    "/content/drive/MyDrive/discovery_results.csv"
)

https://thedonkeysanctuary.org.uk
Processing: BREACH FARM ENERGY STORAGE LIMITED
Processing: RANKSBOROUGH SOLAR LIMITED
Processing: CARRINGTON POWER LIMITED
Processing: MONETS GARDEN BATTERY LTD
Processing: CORRIEGARTH WIND ENERGY LIMITED
Processing: PELAGIC ENERGY DEVELOPMENT LTD
Processing: THAMESWEY ENERGY LIMITED
Processing: HOOTON BIO POWER LIMITED
Processing: ESB SOLAR (NORTHERN IRELAND) LIMITED
Processing: SPRING DEV 02 LIMITED
Processing: BALLYMORE DEANSTON LIMITED
Processing: SEVILLE DEVELOPMENTS LIMITED
Processing: WILLMOTT DIXON FM LIMITED
Processing: NY HIGHWAYS LIMITED
Processing: CANARY WHARF CONTRACTORS (B3 HOTEL) LIMITED
Processing: REEF GROUP LIMITED
Processing: BROADGATE (PHC 2) LIMITED
Processing: CRUDEN BUILDING (SCOTLAND) LIMITED
Processing: OWL HOMES LIMITED
Processing: GHL (ASHFORD) LIMITED
Processing: LUCKNAM PARK HOTELS LIMITED
Processing: DAKOTA HOSPITALITY LIMITED
Processing: MANDEVILLE HOTEL LIMITED
Processing: HILTON UK MADISON SQUARE LIMITED
Processing: TH

In [10]:
# Diagnosing the unexpected 98/100 no_website_found result:
# re-testing 5 queries in a row to check for IP-level blocking
test_companies = ["THE DONKEY SANCTUARY", "RELATE NORTHERN IRELAND", "PBE GROUP LTD",
                   "CHARITIES TRUST", "GROUNDWORK YORKSHIRE LIMITED"]

for name in test_companies:
    query = f"{name} official website"
    resp = requests.get("https://html.duckduckgo.com/html/", params={"q": query},
                         headers=HEADERS, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")
    num_results = len(soup.select("a.result__a"))
    print(f"{name}: status={resp.status_code}, results_found={num_results}, response_length={len(resp.text)}")
    time.sleep(3)

THE DONKEY SANCTUARY: status=202, results_found=0, response_length=14280
RELATE NORTHERN IRELAND: status=202, results_found=0, response_length=14292
PBE GROUP LTD: status=202, results_found=0, response_length=14266
CHARITIES TRUST: status=202, results_found=0, response_length=14269
GROUNDWORK YORKSHIRE LIMITED: status=202, results_found=0, response_length=14302


In [11]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

IGNORE_DOMAINS = [
    "find-and-update.company-information.service.gov.uk",
    "endole.co.uk", "companieshouse.gov.uk", "vat-search.co.uk",
    "vatverifier.com", "linkedin.com", "facebook.com", "wikipedia.org",
    "bloomberg.com", "opencorporates.com", "creditsafe.com", "bing.com",
    "companieslist.co.uk", "ukcompanydir.com",
]

VAT_PATTERN = re.compile(r"\bGB\s?(\d{3})\s?(\d{4})\s?(\d{2})(\d{3})?\b", re.IGNORECASE)
CANDIDATE_PAGES = ["", "/terms", "/terms-and-conditions", "/legal", "/legal-info",
                    "/about", "/contact", "/privacy-policy"]


def find_company_website_bing(company_name: str):
    query = f"{company_name} official website"
    try:
        resp = requests.get(
            "https://www.bing.com/search",
            params={"q": query}, headers=HEADERS, timeout=10
        )
        if resp.status_code != 200:
            print(f"  [WARNING] status={resp.status_code}")
            return None

        soup = BeautifulSoup(resp.text, "html.parser")
        for result in soup.select("li.b_algo h2 a"):
            href = result.get("href", "")
            if href.startswith("http") and not any(d in href for d in IGNORE_DOMAINS):
                domain = re.match(r"https?://(?:www\.)?([^/]+)", href)
                if domain:
                    return f"https://{domain.group(1)}"
    except requests.RequestException as e:
        print(f"  [ERROR] {e}")
    return None

print(find_company_website_bing("THE DONKEY SANCTUARY"))

None


In [12]:
query = "THE DONKEY SANCTUARY official website"
resp = requests.get("https://www.bing.com/search", params={"q": query}, headers=HEADERS, timeout=10)

print("Status code:", resp.status_code)
print("Response length:", len(resp.text))
print("\nFirst 500 chars:")
print(resp.text[:500])

soup = BeautifulSoup(resp.text, "html.parser")
print("\nNumber of results with the current selector (li.b_algo h2 a):", len(soup.select("li.b_algo h2 a")))

print("\nAll links on the page containing 'http' (first 15):")
count = 0
for a in soup.find_all("a", href=True):
    if a["href"].startswith("http") and count < 15:
        print(a["href"])
        count += 1

Status code: 200
Response length: 115353

First 500 chars:
<!DOCTYPE html><html dir="ltr" lang="zh" xml:lang="zh" xmlns="http://www.w3.org/1999/xhtml" xmlns:Web="http://schemas.live.com/Web/"><script type="text/javascript" nonce="P9THfZckAzt6C5eWl7vHvxleYYneNd7V2Bp+7lrXdJY=" >//<![CDATA[
window.si_ST=new Date
//]]></script><head><!--pc--><title>THE DONKEY SANCTUARY official website - 搜尋</title><meta content="text/html; charset=utf-8" http-equiv="content-type" /><meta name="referrer" content="origin-when-cross-origin" /><meta property="og:description" 

Number of results with the current selector (li.b_algo h2 a): 7

All links on the page containing 'http' (first 15):
https://www.bing.com/ck/a?!&&p=9bcad88acf4d765f926d44f299e1188ee35df6167c55f1904b4face62e9385c1JmltdHM9MTc4NjgzODQwMA&ptn=3&ver=2&hsh=4&fclid=123404e4-fc61-602b-1a7d-135cfd826130&u=a1L2ltYWdlcy9zZWFyY2g_cT1USEUrRE9OS0VZK1NBTkNUVUFSWStvZmZpY2lhbCt3ZWJzaXRlJkZPUk09SERSU0My&ntb=1
https://www.bing.com/ck/a?!&&p=c42a8053d5de0cf

In [13]:
for a in soup.find_all("a", href=True):
    if "bing.com/ck/a" in a["href"]:
        print(a["href"])
        print("---")
        break

https://www.bing.com/ck/a?!&&p=9bcad88acf4d765f926d44f299e1188ee35df6167c55f1904b4face62e9385c1JmltdHM9MTc4NjgzODQwMA&ptn=3&ver=2&hsh=4&fclid=123404e4-fc61-602b-1a7d-135cfd826130&u=a1L2ltYWdlcy9zZWFyY2g_cT1USEUrRE9OS0VZK1NBTkNUVUFSWStvZmZpY2lhbCt3ZWJzaXRlJkZPUk09SERSU0My&ntb=1
---


In [15]:
import base64
from urllib.parse import urlparse, parse_qs, unquote

# Decoding the actual destination URL hidden in Bing's redirect link
sample_href = "https://www.bing.com/ck/a?!&&p=8ecb4dcc7fb10d375092265461f03c90050e88105ac5b7c21ed0c50b0b212687JmltdHM9MTc4NjgzODQwMA&ptn=3&ver=2&hsh=4&fclid=23ffc47d-0149-635d-2c92-d3ca00066255&u=a1L2ltYWdlcy9zZWFyY2g_cT1USEUrRE9OS0VZK1NBTkNUVUFSWStvZmZpY2lhbCt3ZWJzaXRlJkZPUk09SERSU0My&ntb=1"

parsed = urlparse(sample_href)
qs = parse_qs(parsed.query)
u_param = qs["u"][0]

# Bing prefixes the base64 payload with "a1" before the actual encoded string
encoded_part = u_param[2:] if u_param.startswith("a1") else u_param
# Add padding if needed for valid base64
padded = encoded_part + "=" * (-len(encoded_part) % 4)
decoded_url = base64.urlsafe_b64decode(padded).decode("utf-8", errors="replace")

print("Decoded destination URL:", decoded_url)

Decoded destination URL: /images/search?q=THE+DONKEY+SANCTUARY+official+website&FORM=HDRSC2
